
# Round4 CLEAN — YOLO 대피안내도 객체 탐지

이 노트북은 **위에서 아래로 순서대로 실행**하면 됩니다.

### 고정 클래스 순서 — 절대 변경 금지

```text
0 exit
1 stair
2 elevator
3 extinguisher
4 hydrant
5 you_are_here
6 door
7 room
```

### Round4 원칙
- 시작 모델: **Round2 best.pt**
- Round2 YOLO의 예측을 다시 pseudo-label로 사용하지 않음
- 외부 teacher: **Grounding DINO**
- 핵심 클래스: `exit / stair / you_are_here`
- teacher 결과는 전체 pseudo-label 이미지가 아니라 **객체 crop** 위주로 사용
- `stair` synthetic을 가장 많이 생성
- validation은 **사람 GT**만 사용
- Stage A/B가 Round2보다 나빠지면 **Round2로 자동 rollback**
- Colab 절약 설정: `imgsz=768, batch=2, workers=1, cache=False`

> 처음에는 **Cell 1부터 순서대로 실행**하세요. 셀 내부 일부를 따로 수정할 필요가 없습니다.


In [ ]:

# ============================================================
# Cell 1 — 설치 + Drive 연결 + 기본 설정
# ============================================================

!pip -q install "ultralytics>=8.3,<9" "transformers>=4.52,<6" accelerate opencv-python-headless pyyaml tqdm

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, gc, json, random, shutil, yaml, math
import numpy as np
import cv2
import torch

# ------------------------------------------------------------
# 절대 변경 금지
# ------------------------------------------------------------
CLASS_NAMES = [
    'exit',
    'stair',
    'elevator',
    'extinguisher',
    'hydrant',
    'you_are_here',
    'door',
    'room',
]
CORE_IDS = [0, 1, 5]

assert CLASS_NAMES == [
    'exit','stair','elevator','extinguisher',
    'hydrant','you_are_here','door','room'
]

# ------------------------------------------------------------
# 현재 확인된 Round2 best.pt — 이미 채워둠
# ------------------------------------------------------------
ROUND2_BEST = Path(
    '/content/drive/MyDrive/evacuation_checkpoints/'
    'round2/evac_round2_stage_a/weights/best.pt'
)

# Round4에서 새로 수집한 실제 대피도 이미지 폴더.
# 없으면 teacher 단계는 자동 skip됩니다.
ROUND4_UNLABELED = Path('/content/drive/MyDrive/evacuation_yolo/round4_unlabeled')

# 출력 폴더
OUTPUT_ROOT = Path('/content/drive/MyDrive/evacuation_yolo/round4_clean')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# 메모리 절약
IMGSZ = 768
BATCH = 2
WORKERS = 1
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
SEED = 3407
random.seed(SEED)
np.random.seed(SEED)

print('Round2 best :', ROUND2_BEST)
print('Exists      :', ROUND2_BEST.exists())
print('Unlabeled   :', ROUND4_UNLABELED)
print('Output      :', OUTPUT_ROOT)
print('Device      :', DEVICE)

if not ROUND2_BEST.exists():
    raise FileNotFoundError(f'Round2 best.pt가 없습니다: {ROUND2_BEST}')

print('\n✅ Cell 1 OK')


In [ ]:

# ============================================================
# Cell 2 — Round2 사람이 만든 GT 데이터셋 자동 복구
#
# 1순위: Round2 training run의 args.yaml → 당시 data.yaml 복구
# 2순위: Drive 안의 YOLO data.yaml 후보 탐색
# 3순위: 흔한 YOLO 폴더 구조 탐색
#
# 성공하면 ROUND2_SPLITS가 자동으로 만들어집니다.
# ============================================================

from ultralytics import YOLO

IMAGE_EXTS = {'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff','.avif','.heic','.heif'}


def find_images(p):
    p = Path(p)
    if not p.exists():
        return []
    return sorted(x for x in p.rglob('*') if x.is_file() and x.suffix.lower() in IMAGE_EXTS)


def count_images(p):
    return len(find_images(p))


def count_labels(p):
    p = Path(p)
    if not p.exists():
        return 0
    return len(list(p.rglob('*.txt')))


def resolve_yaml_path(value, yaml_file, dataset_root=None):
    if value is None:
        return None
    if isinstance(value, list):
        return None
    p = Path(str(value))
    if p.is_absolute():
        return p
    if dataset_root is not None:
        q = Path(dataset_root) / p
        if q.exists():
            return q
    return yaml_file.parent / p


def infer_label_dir(image_dir):
    image_dir = Path(image_dir)
    s = str(image_dir)

    # root/train/images -> root/train/labels
    if image_dir.name == 'images':
        q = image_dir.parent / 'labels'
        if q.exists():
            return q

    # root/images/train -> root/labels/train
    parts = list(image_dir.parts)
    if 'images' in parts:
        idx = len(parts) - 1 - parts[::-1].index('images')
        parts[idx] = 'labels'
        q = Path(*parts)
        if q.exists():
            return q
    return None


def dataset_from_yaml(yaml_path):
    yaml_path = Path(yaml_path)
    if not yaml_path.exists():
        return None
    try:
        d = yaml.safe_load(yaml_path.read_text(encoding='utf-8')) or {}
    except Exception:
        return None

    names = d.get('names')
    if isinstance(names, dict):
        try:
            names_list = [names[i] if i in names else names[str(i)] for i in range(len(names))]
        except Exception:
            names_list = list(names.values())
    elif isinstance(names, list):
        names_list = names
    else:
        names_list = None

    # 클래스 이름이 명시된 YAML이면 정확히 8개 순서를 요구
    if names_list is not None and list(names_list) != CLASS_NAMES:
        return None

    root = d.get('path')
    if root is not None:
        root = Path(str(root))
        if not root.is_absolute():
            root = yaml_path.parent / root

    out = {'yaml': yaml_path}
    for split in ['train','val','test']:
        ip = resolve_yaml_path(d.get(split), yaml_path, root)
        if ip is not None:
            lp = infer_label_dir(ip)
            out[split] = {'images': ip, 'labels': lp}
        else:
            out[split] = None
    return out


def valid_dataset(ds):
    if not ds or not ds.get('train') or not ds.get('val'):
        return False
    tr, va = ds['train'], ds['val']
    if tr['labels'] is None or va['labels'] is None:
        return False
    return (
        count_images(tr['images']) > 0 and count_labels(tr['labels']) > 0 and
        count_images(va['images']) > 0 and count_labels(va['labels']) > 0
    )


def common_layout_candidate(root):
    root = Path(root)
    layouts = [
        {
            'train': (root/'train/images', root/'train/labels'),
            'val':   (root/'val/images',   root/'val/labels'),
            'test':  (root/'test/images',  root/'test/labels'),
        },
        {
            'train': (root/'images/train', root/'labels/train'),
            'val':   (root/'images/val',   root/'labels/val'),
            'test':  (root/'images/test',  root/'labels/test'),
        },
    ]
    for layout in layouts:
        if (count_images(layout['train'][0]) > 0 and count_labels(layout['train'][1]) > 0 and
            count_images(layout['val'][0]) > 0 and count_labels(layout['val'][1]) > 0):
            return {
                'yaml': None,
                'train': {'images':layout['train'][0], 'labels':layout['train'][1]},
                'val': {'images':layout['val'][0], 'labels':layout['val'][1]},
                'test': {'images':layout['test'][0], 'labels':layout['test'][1]},
            }
    return None

# ------------------------------------------------------------
# Round2 모델 클래스 순서 검증
# ------------------------------------------------------------
model = YOLO(str(ROUND2_BEST))
model_names = model.names
if isinstance(model_names, dict):
    model_names = [model_names[i] for i in range(len(model_names))]
else:
    model_names = list(model_names)

del model

if model_names != CLASS_NAMES:
    raise RuntimeError(f'클래스 순서 불일치: {model_names}')

print('✅ Round2 model class order OK')

# ------------------------------------------------------------
# 1) args.yaml에서 원래 dataset YAML 찾기
# ------------------------------------------------------------
candidates = []
run_root = ROUND2_BEST.parent.parent
args_yaml = run_root / 'args.yaml'

if args_yaml.exists():
    try:
        args = yaml.safe_load(args_yaml.read_text(encoding='utf-8')) or {}
        data_ref = args.get('data')
        if data_ref:
            dp = Path(str(data_ref))
            if dp.exists() and dp.suffix.lower() in {'.yaml','.yml'}:
                ds = dataset_from_yaml(dp)
                if valid_dataset(ds):
                    candidates.append(('Round2 args.yaml', ds))
            else:
                # args.yaml에 과거 /content 경로가 남아 있을 경우 파일명으로 Drive 탐색
                name = dp.name
                for yp in Path('/content/drive/MyDrive').rglob(name):
                    ds = dataset_from_yaml(yp)
                    if valid_dataset(ds):
                        candidates.append(('Round2 args.yaml basename recovery', ds))
    except Exception as e:
        print('args.yaml 해석 skip:', e)

# ------------------------------------------------------------
# 2) Drive의 YAML 탐색
# ------------------------------------------------------------
if not candidates:
    for pattern in ['*.yaml','*.yml']:
        for yp in Path('/content/drive/MyDrive').rglob(pattern):
            try:
                ds = dataset_from_yaml(yp)
                if valid_dataset(ds):
                    candidates.append(('Drive YAML', ds))
            except Exception:
                pass

# ------------------------------------------------------------
# 3) 흔한 폴더 구조 탐색
# ------------------------------------------------------------
if not candidates:
    seen = set()
    drive_root = Path('/content/drive/MyDrive')
    for train_dir in drive_root.rglob('train'):
        for root in [train_dir.parent, train_dir.parent.parent]:
            if root in seen:
                continue
            seen.add(root)
            ds = common_layout_candidate(root)
            if valid_dataset(ds):
                candidates.append(('Folder structure', ds))

# 중복 제거
uniq=[]; sigs=set()
for source,ds in candidates:
    sig=(str(ds['train']['images']),str(ds['val']['images']))
    if sig not in sigs:
        sigs.add(sig); uniq.append((source,ds))
candidates=uniq

if not candidates:
    print('\n❌ Round2 HUMAN-GT train/val 데이터셋을 자동으로 찾지 못했습니다.')
    print('Round4 학습은 사람 GT validation 없이 진행하면 안 되므로 여기서 안전하게 중단합니다.')
    print('\n필요한 것은 딱 하나입니다: 예전에 Round2에서 사용한 data.yaml 파일 또는 그 dataset 폴더.')
    print('그 파일/폴더를 Drive에 올린 뒤 Cell 2를 다시 실행하세요.')
    ROUND2_SPLITS = None
else:
    # Round2 args.yaml 출처를 우선, 그 외에는 val 크기가 큰 후보 우선
    candidates.sort(key=lambda z: (0 if 'args.yaml' in z[0] else 1, -count_images(z[1]['val']['images'])))
    source, ROUND2_SPLITS = candidates[0]

    # test가 없으면 독립 test는 없다고 명시하고 val을 test로 복제하지 않음
    if ROUND2_SPLITS.get('test'):
        ti = ROUND2_SPLITS['test']['images']
        tl = ROUND2_SPLITS['test']['labels']
        if tl is None or count_images(ti) == 0 or count_labels(tl) == 0:
            ROUND2_SPLITS['test'] = None

    print('\n✅ Round2 HUMAN-GT dataset 자동 선택')
    print('Source:', source)
    for split in ['train','val','test']:
        x = ROUND2_SPLITS.get(split)
        if not x:
            print(f'{split:5s}: 없음')
        else:
            print(f"{split:5s}: {count_images(x['images'])} images | {x['images']}")
            print(f"       {count_labels(x['labels'])} labels | {x['labels']}")

    print('\n✅ Cell 2 OK — 다음 셀로 진행하세요.')


In [ ]:

# ============================================================
# Cell 3 — Round2 GT crop + Grounding DINO teacher crop
# ============================================================

if ROUND2_SPLITS is None:
    print('❌ Cell 2에서 HUMAN-GT dataset을 찾지 못했습니다. Cell 3은 실행하지 않습니다.')
else:
    from collections import Counter
    from PIL import Image
    from tqdm.auto import tqdm
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

    CROP_GT = OUTPUT_ROOT / 'round2_gt_crops'
    CROP_TEACHER = OUTPUT_ROOT / 'teacher_crops'
    for root in [CROP_GT, CROP_TEACHER]:
        if root.exists(): shutil.rmtree(root)
        for cid in CORE_IDS: (root/str(cid)).mkdir(parents=True, exist_ok=True)

    def image_to_label(ip, image_root, label_root):
        return (Path(label_root) / Path(ip).relative_to(image_root)).with_suffix('.txt')

    def read_yolo(lp):
        rows=[]
        if not Path(lp).exists(): return rows
        for line in Path(lp).read_text(encoding='utf-8', errors='ignore').splitlines():
            p=line.strip().split()
            if len(p)>=5:
                try:
                    cid=int(float(p[0])); vals=list(map(float,p[1:5]))
                    if 0<=cid<8: rows.append((cid,*vals))
                except: pass
        return rows

    # --------------------------------------------------------
    # 3-A) HUMAN GT crop 추출
    # --------------------------------------------------------
    counts=Counter()
    tri=ROUND2_SPLITS['train']['images']; trl=ROUND2_SPLITS['train']['labels']
    for ip in tqdm(find_images(tri), desc='Round2 GT crops'):
        img=cv2.imread(str(ip))
        if img is None: continue
        H,W=img.shape[:2]
        lp=image_to_label(ip,tri,trl)
        for j,row in enumerate(read_yolo(lp)):
            cid,xc,yc,bw,bh=row
            if cid not in CORE_IDS: continue
            x1=(xc-bw/2)*W; y1=(yc-bh/2)*H; x2=(xc+bw/2)*W; y2=(yc+bh/2)*H
            px=max(3,int((x2-x1)*.12)); py=max(3,int((y2-y1)*.12))
            x1=max(0,int(x1)-px); y1=max(0,int(y1)-py); x2=min(W,int(x2)+px); y2=min(H,int(y2)+py)
            crop=img[y1:y2,x1:x2]
            if crop.size==0 or min(crop.shape[:2])<5: continue
            cv2.imwrite(str(CROP_GT/str(cid)/f'{ip.stem}_{j:03d}.png'),crop)
            counts[cid]+=1

    print('\nHUMAN-GT crop bank:')
    for cid in CORE_IDS: print(cid, CLASS_NAMES[cid], counts[cid])

    # --------------------------------------------------------
    # 3-B) Grounding DINO — 새 real 이미지가 있을 때만 실행
    # --------------------------------------------------------
    teacher_images=find_images(ROUND4_UNLABELED)
    print('\nRound4 unlabeled images:', len(teacher_images))

    if len(teacher_images)==0:
        print('ℹ️ 새 unlabeled real 이미지가 없어서 Grounding DINO 단계는 skip합니다.')
        print('   Round2 HUMAN-GT crop + stair fallback synthetic만으로 진행합니다.')
    else:
        TEACHER_ID='IDEA-Research/grounding-dino-tiny'
        prompts=[
            'emergency exit sign','exit sign','green emergency exit symbol','evacuation exit pictogram',
            'stairs icon','staircase symbol','stairway symbol','stair steps on floor plan',
            'you are here marker','current location marker','location marker on evacuation map','you are here symbol'
        ]
        limits={0:150,1:180,5:150}
        thresholds={0:.30,1:.27,5:.30}
        strong={0:.55,1:.50,5:.55}

        def label_cls(t):
            t=str(t).lower()
            if 'stair' in t: return 1
            if 'you are here' in t or 'current location' in t or 'location marker' in t: return 5
            if 'exit' in t: return 0
            return None

        def iou(a,b):
            ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
            x1,y1=max(ax1,bx1),max(ay1,by1); x2,y2=min(ax2,bx2),min(ay2,by2)
            inter=max(0,x2-x1)*max(0,y2-y1)
            union=max(0,ax2-ax1)*max(0,ay2-ay1)+max(0,bx2-bx1)*max(0,by2-by1)-inter
            return inter/union if union>0 else 0

        def valid_box(b,W,H):
            x1,y1,x2,y2=b; bw=x2-x1; bh=y2-y1
            if bw<6 or bh<6:return False
            ar=(bw*bh)/(W*H)
            if ar<0.000005 or ar>0.12:return False
            return max(bw/max(bh,1),bh/max(bw,1))<=9

        def views(img):
            W,H=img.size; out=[('full',img,0,0)]
            tw=min(W,max(256,int(W*.58))); th=min(H,max(256,int(H*.58)))
            for k,(x,y) in enumerate([(x,y) for y in sorted(set([0,max(0,H-th)])) for x in sorted(set([0,max(0,W-tw)]))]):
                out.append((f'tile{k}',img.crop((x,y,min(W,x+tw),min(H,y+th))),x,y))
            return out

        device='cuda' if torch.cuda.is_available() else 'cpu'
        proc=AutoProcessor.from_pretrained(TEACHER_ID)
        teacher=AutoModelForZeroShotObjectDetection.from_pretrained(TEACHER_ID).to(device).eval()
        saved={0:0,1:0,5:0}

        for ip in tqdm(teacher_images,desc='Grounding DINO'):
            if all(saved[c]>=limits[c] for c in CORE_IDS): break
            img=Image.open(ip).convert('RGB'); W,H=img.size; preds=[]
            for vn,vi,ox,oy in views(img):
                inp=proc(images=vi,text=[prompts],return_tensors='pt')
                inp={k:(v.to(device) if hasattr(v,'to') else v) for k,v in inp.items()}
                with torch.inference_mode(): out=teacher(**inp)
                res=proc.post_process_grounded_object_detection(out,inp['input_ids'],threshold=.22,text_threshold=.20,target_sizes=[(vi.height,vi.width)])[0]
                for box,score,label in zip(res['boxes'],res['scores'],res['labels']):
                    cid=label_cls(label)
                    if cid not in CORE_IDS: continue
                    x1,y1,x2,y2=box.detach().cpu().tolist(); b=[x1+ox,y1+oy,x2+ox,y2+oy]
                    if valid_box(b,W,H): preds.append({'cid':cid,'score':float(score.detach().cpu()),'box':b,'view':vn})
                del out,inp
                if torch.cuda.is_available(): torch.cuda.empty_cache()

            accepted=[]
            for cid in CORE_IDS:
                grp=sorted([p for p in preds if p['cid']==cid],key=lambda z:z['score'],reverse=True)
                for i,p in enumerate(grp):
                    support=[p]+[q for q in grp[i+1:] if q['view']!=p['view'] and iou(p['box'],q['box'])>=.25]
                    if len(support)>=2 and np.mean([x['score'] for x in support])>=thresholds[cid]:
                        w=np.array([x['score'] for x in support]); boxes=np.array([x['box'] for x in support])
                        accepted.append((cid,float(w.mean()),np.average(boxes,axis=0,weights=w).tolist()))
                    elif p['score']>=strong[cid]: accepted.append((cid,p['score'],p['box']))

            arr=np.array(img)
            final=[]
            for cid in CORE_IDS:
                keep=[]
                for x in sorted([a for a in accepted if a[0]==cid],key=lambda z:z[1],reverse=True):
                    if all(iou(x[2],q[2])<=.45 for q in keep): keep.append(x)
                final+=keep
            for cid,score,b in final:
                if saved[cid]>=limits[cid]: continue
                x1,y1,x2,y2=b; px=max(4,int((x2-x1)*.15)); py=max(4,int((y2-y1)*.15))
                x1=max(0,int(x1)-px); y1=max(0,int(y1)-py); x2=min(W,int(x2)+px); y2=min(H,int(y2)+py)
                crop=arr[y1:y2,x1:x2]
                if crop.size:
                    Image.fromarray(crop).save(CROP_TEACHER/str(cid)/f'{ip.stem}_{saved[cid]:04d}.png')
                    saved[cid]+=1
            del img,arr; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

        del teacher,proc; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print('\nTeacher crops:', {CLASS_NAMES[k]:v for k,v in saved.items()})

    print('\n✅ Cell 3 OK')


In [ ]:

# ============================================================
# Cell 4 — stair 중심 synthetic 생성 + Round4 dataset 구성
# ============================================================

if ROUND2_SPLITS is None:
    print('❌ Cell 2 미완료')
else:
    from collections import Counter
    from tqdm.auto import tqdm

    SYNTH_ROOT=OUTPUT_ROOT/'core_synthetic'
    if SYNTH_ROOT.exists(): shutil.rmtree(SYNTH_ROOT)
    (SYNTH_ROOT/'images').mkdir(parents=True); (SYNTH_ROOT/'labels').mkdir(parents=True)

    TARGETS={0:180,1:420,5:220}
    SIZE=768

    def floorplan():
        img=np.full((SIZE,SIZE,3),random.randint(238,255),np.uint8)
        for _ in range(random.randint(12,28)):
            x1=random.randint(30,SIZE-100); y1=random.randint(30,SIZE-100)
            if random.random()<.5: x2=min(SIZE-20,x1+random.randint(80,300)); y2=y1
            else: x2=x1; y2=min(SIZE-20,y1+random.randint(80,300))
            g=random.randint(120,210); cv2.line(img,(x1,y1),(x2,y2),(g,g,g),random.randint(1,3))
        for _ in range(random.randint(3,10)):
            x1=random.randint(20,SIZE-200); y1=random.randint(20,SIZE-200)
            x2=min(SIZE-20,x1+random.randint(80,240)); y2=min(SIZE-20,y1+random.randint(60,200))
            g=random.randint(150,220); cv2.rectangle(img,(x1,y1),(x2,y2),(g,g,g),1)
        return img

    def fallback(cid,s=96):
        p=np.full((s,s,3),245,np.uint8)
        if cid==1:
            # 여러 방향/형태를 augmentation에서 회전해 사용
            for i in range(6):
                x1=15+i*9; y1=s-15-i*9; x2=15+(i+1)*9; y2=s-15-(i+1)*9
                cv2.line(p,(x1,y1),(x2,y1),(20,20,20),4); cv2.line(p,(x2,y1),(x2,y2),(20,20,20),4)
        elif cid==0:
            cv2.rectangle(p,(10,20),(s-10,s-20),(60,160,70),-1)
            cv2.arrowedLine(p,(25,s//2),(s-25,s//2),(255,255,255),6,tipLength=.3)
        else:
            c=(s//2,s//2); cv2.circle(p,c,s//5,(20,20,220),-1); cv2.circle(p,c,s//3,(20,20,220),3)
        return p

    banks={}
    for cid in CORE_IDS:
        files=[]
        for root in [OUTPUT_ROOT/'round2_gt_crops',OUTPUT_ROOT/'teacher_crops']:
            d=root/str(cid)
            if d.exists(): files += [p for p in d.glob('*') if p.suffix.lower() in IMAGE_EXTS]
        banks[cid]=files
        print('bank',cid,CLASS_NAMES[cid],len(files))

    synth_counts=Counter()
    for focus in CORE_IDS:
        for i in tqdm(range(TARGETS[focus]),desc=f'synth {CLASS_NAMES[focus]}'):
            canvas=floorplan(); labels=[]
            classes=[focus]+[random.choice(CORE_IDS) for _ in range(random.randint(0,3))]
            for cid in classes:
                patch=cv2.imread(str(random.choice(banks[cid]))) if banks[cid] else fallback(cid)
                if patch is None: patch=fallback(cid)
                patch=cv2.convertScaleAbs(patch,alpha=random.uniform(.82,1.18),beta=random.randint(-18,18))
                if random.random()<.2: patch=cv2.GaussianBlur(patch,(3,3),0)
                h,w=patch.shape[:2]
                M=cv2.getRotationMatrix2D((w/2,h/2),random.uniform(-15,15),1)
                patch=cv2.warpAffine(patch,M,(w,h),borderValue=(245,245,245))
                h,w=patch.shape[:2]; max_side=random.randint(24,110); scale=max_side/max(h,w)
                nw=max(12,int(w*scale)); nh=max(12,int(h*scale))
                if nw>=SIZE-20 or nh>=SIZE-20: continue
                patch=cv2.resize(patch,(nw,nh),interpolation=cv2.INTER_AREA)
                x1=random.randint(10,SIZE-nw-10); y1=random.randint(10,SIZE-nh-10); x2=x1+nw; y2=y1+nh
                canvas[y1:y2,x1:x2]=patch
                xc=((x1+x2)/2)/SIZE; yc=((y1+y2)/2)/SIZE; bw=(x2-x1)/SIZE; bh=(y2-y1)/SIZE
                labels.append(f'{cid} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
                synth_counts[cid]+=1
            stem=f'core_{focus}_{i:05d}'
            cv2.imwrite(str(SYNTH_ROOT/'images'/f'{stem}.jpg'),canvas)
            (SYNTH_ROOT/'labels'/f'{stem}.txt').write_text('\n'.join(labels),encoding='utf-8')

    print('Synthetic objects:',{CLASS_NAMES[k]:synth_counts[k] for k in CORE_IDS})

    # --------------------------------------------------------
    # Dataset build
    # HUMAN GT train + core oversampling + Round4 synthetic
    # val/test는 HUMAN GT만
    # --------------------------------------------------------
    DS=OUTPUT_ROOT/'dataset'
    if DS.exists(): shutil.rmtree(DS)
    for split in ['train','val','test']:
        (DS/split/'images').mkdir(parents=True,exist_ok=True)
        (DS/split/'labels').mkdir(parents=True,exist_ok=True)

    def copy_pair(ip,lp,di,dl,stem):
        if not Path(lp).exists(): return False
        shutil.copy2(ip,di/f'{stem}{Path(ip).suffix.lower()}')
        shutil.copy2(lp,dl/f'{stem}.txt')
        return True

    tri=ROUND2_SPLITS['train']['images']; trl=ROUND2_SPLITS['train']['labels']
    core_pairs=[]; n=0
    for i,ip in enumerate(find_images(tri)):
        lp=image_to_label(ip,tri,trl)
        if copy_pair(ip,lp,DS/'train/images',DS/'train/labels',f'r2_{i:05d}'):
            n+=1
            if any(r[0] in CORE_IDS for r in read_yolo(lp)): core_pairs.append((ip,lp))

    # HUMAN GT core image 1회 추가 복제
    for j,(ip,lp) in enumerate(core_pairs):
        copy_pair(ip,lp,DS/'train/images',DS/'train/labels',f'r2core_{j:05d}')

    # synthetic 최대 820장
    simgs=find_images(SYNTH_ROOT/'images'); random.shuffle(simgs)
    for i,ip in enumerate(simgs[:820]):
        lp=SYNTH_ROOT/'labels'/f'{ip.stem}.txt'
        copy_pair(ip,lp,DS/'train/images',DS/'train/labels',f'r4s_{i:05d}')

    # HUMAN GT val/test
    for split in ['val','test']:
        x=ROUND2_SPLITS.get(split)
        if not x: continue
        for i,ip in enumerate(find_images(x['images'])):
            lp=image_to_label(ip,x['images'],x['labels'])
            copy_pair(ip,lp,DS/f'{split}/images',DS/f'{split}/labels',f'{split}_{i:05d}')

    trn=count_images(DS/'train/images'); van=count_images(DS/'val/images'); ten=count_images(DS/'test/images')
    print('\nRound4 dataset: train=',trn,'val=',van,'test=',ten)
    if van==0:
        raise RuntimeError('안전 중단: HUMAN-GT validation이 0장입니다. 학습하지 않습니다.')

    data_yaml=OUTPUT_ROOT/'round4_data.yaml'
    data={
        'path':str(DS),
        'train':'train/images',
        'val':'val/images',
        'names':{i:n for i,n in enumerate(CLASS_NAMES)}
    }
    if ten>0: data['test']='test/images'
    data_yaml.write_text(yaml.safe_dump(data,allow_unicode=True,sort_keys=False),encoding='utf-8')
    print('data.yaml:',data_yaml)
    print('\n✅ Cell 4 OK')


In [ ]:

# ============================================================
# Cell 5 — Round2 baseline 평가 → Stage A → Stage B
#          → regression guard → PT + ONNX
# ============================================================

if ROUND2_SPLITS is None:
    print('❌ Cell 2 미완료')
else:
    from ultralytics import YOLO

    DATA_YAML=OUTPUT_ROOT/'round4_data.yaml'
    RUNS=OUTPUT_ROOT/'runs'
    RUNS.mkdir(parents=True,exist_ok=True)

    def metric_dict(r):
        b=r.box
        out={
            'precision':float(b.mp),
            'recall':float(b.mr),
            'map50':float(b.map50),
            'map50_95':float(b.map),
            'per_class':{}
        }
        for i,name in enumerate(CLASS_NAMES):
            try:
                cr=b.class_result(i)
                out['per_class'][name]={
                    'precision':float(cr[0]),'recall':float(cr[1]),
                    'map50':float(cr[2]),'map50_95':float(cr[3])
                }
            except Exception:
                out['per_class'][name]={}
        return out

    def validate(weights):
        m=YOLO(str(weights))
        r=m.val(data=str(DATA_YAML),imgsz=IMGSZ,batch=BATCH,workers=WORKERS,device=DEVICE,split='val',verbose=False)
        return metric_dict(r)

    print('===== Round2 baseline on HUMAN-GT val =====')
    base=validate(ROUND2_BEST)
    print(base)

    common=dict(
        data=str(DATA_YAML),imgsz=IMGSZ,batch=BATCH,workers=WORKERS,
        cache=False,device=DEVICE,amp=True,cos_lr=True,close_mosaic=8,
        project=str(RUNS),verbose=True
    )

    print('\n===== Stage A =====')
    ma=YOLO(str(ROUND2_BEST))
    ma.train(**common,name='stage_a',epochs=25,lr0=0.0005,lrf=0.1,freeze=10,patience=10)
    A=RUNS/'stage_a/weights/best.pt'
    am=validate(A)

    print('\n===== Stage B =====')
    mb=YOLO(str(A))
    mb.train(**common,name='stage_b',epochs=35,lr0=0.00015,lrf=0.05,freeze=0,patience=12)
    B=RUNS/'stage_b/weights/best.pt'
    bm=validate(B)

    def guard(metrics):
        overall_ratio=metrics['map50']/max(base['map50'],1e-9)
        ratios=[]
        for name in ['exit','stair','you_are_here']:
            b=base['per_class'].get(name,{}).get('map50')
            c=metrics['per_class'].get(name,{}).get('map50')
            if b is not None and c is not None and b>0: ratios.append(c/b)
        core_ratio=sum(ratios)/len(ratios) if ratios else overall_ratio
        ok=overall_ratio>=0.97 and core_ratio>=0.95
        core_score=sum(metrics['per_class'].get(n,{}).get('map50',0) or 0 for n in ['exit','stair','you_are_here'])
        return ok,overall_ratio,core_ratio,(core_score,metrics['map50'])

    records=[]
    for name,path,m in [('round2',ROUND2_BEST,base),('stage_a',A,am),('stage_b',B,bm)]:
        if name=='round2': ok,orr,cr,pri=True,1.0,1.0,(sum(m['per_class'].get(n,{}).get('map50',0) or 0 for n in ['exit','stair','you_are_here']),m['map50'])
        else: ok,orr,cr,pri=guard(m)
        records.append({'name':name,'path':str(path),'guard_ok':ok,'overall_ratio':orr,'core_ratio':cr,'priority':list(pri),'metrics':m})

    valid=[x for x in records if x['guard_ok']]
    selected=max(valid,key=lambda x:tuple(x['priority'])) if valid else records[0]

    FINAL_PT=OUTPUT_ROOT/'guarded_best.pt'
    shutil.copy2(selected['path'],FINAL_PT)

    print('\n===== SELECTION =====')
    print('Selected:',selected['name'])
    print('Final PT:',FINAL_PT)

    FINAL_ONNX=None
    try:
        export_model=YOLO(str(FINAL_PT))
        ep=Path(export_model.export(format='onnx',imgsz=IMGSZ,opset=12,simplify=True))
        FINAL_ONNX=OUTPUT_ROOT/'guarded_best.onnx'
        shutil.copy2(ep,FINAL_ONNX)
        print('Final ONNX:',FINAL_ONNX)
    except Exception as e:
        print('⚠️ ONNX export 실패 — PT는 정상 보존됨:',repr(e))

    report={
        'selected':selected['name'],
        'final_pt':str(FINAL_PT),
        'final_onnx':str(FINAL_ONNX) if FINAL_ONNX else None,
        'candidates':records,
        'class_order':CLASS_NAMES,
    }
    (OUTPUT_ROOT/'selection_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')

    print('\n✅ Cell 5 OK')


In [ ]:

# ============================================================
# Cell 6 — 최종 확인
# ============================================================

print('Output:', OUTPUT_ROOT)

files=[
    OUTPUT_ROOT/'guarded_best.pt',
    OUTPUT_ROOT/'guarded_best.onnx',
    OUTPUT_ROOT/'selection_report.json',
    OUTPUT_ROOT/'round4_data.yaml',
]

all_ok=True
for p in files:
    ok=p.exists()
    print(('✅' if ok else '❌'),p.name, ('READY' if ok else 'missing'))
    if p.name!='guarded_best.onnx' and not ok:
        all_ok=False

if all_ok:
    print('\n🎉 Round4 완료')
    print('실제 앱/추론에는 guarded_best.pt를 우선 사용하세요.')
    if not (OUTPUT_ROOT/'guarded_best.onnx').exists():
        print('ONNX만 실패한 상태입니다. PT 모델은 사용할 수 있습니다.')
else:
    print('\n아직 Round4가 끝나지 않았습니다. 위에서 처음 ❌/오류가 난 셀만 확인하세요.')


In [ ]:
import json
from pathlib import Path

report = Path(
    "/content/drive/MyDrive/evacuation_yolo/"
    "round4_clean/selection_report.json"
)

with open(report, "r", encoding="utf-8") as f:
    data = json.load(f)

print(json.dumps(data, indent=2, ensure_ascii=False))